# 准备文档

In [1]:
from langchain.agents import create_agent
from langchain_text_splitters import MarkdownHeaderTextSplitter

# 1.读取文档
with open('./resources/评估.md', 'r', encoding='utf-8') as f:
    doc = f.read()

# 2.切分文档
headers_to_split_on = [
    ("#", "H1"),
    ("##", "H2"),
    ("###", "H3"),
]
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on,
    strip_headers=False # 是否去除文本中的标题，False：不去除
)
# list[Document]
documents = markdown_splitter.split_text(doc)

for i, document in enumerate(documents, start=1):
    document.id = f"doc_{i}"

# 准备知识库

In [2]:
from langchain_ollama import OllamaEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

# 1.初始化向量模型
ollama_embeddings = OllamaEmbeddings(
    model="qwen3-embedding:0.6b",  # 性价比高的模型
    dimensions=1024  # 向量维度，可选：减少维度以节省存储
)

# 2.初始化VectorStore
vector_store = InMemoryVectorStore(ollama_embeddings)

# 3.添加文档
vector_store.add_documents(documents)

['doc_1',
 'doc_2',
 'doc_3',
 'doc_4',
 'doc_5',
 'doc_6',
 'doc_7',
 'doc_8',
 'doc_9',
 'doc_10',
 'doc_11',
 'doc_12',
 'doc_13',
 'doc_14',
 'doc_15',
 'doc_16',
 'doc_17',
 'doc_18',
 'doc_19',
 'doc_20',
 'doc_21']

# 2-Step模式

In [3]:
from langchain.agents.middleware import dynamic_prompt, ModelRequest
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()

# 0.定义动态提示词中间件
@dynamic_prompt
def rag_prompt(request: ModelRequest):
    # 1.拿到用户问题，也就是消息列表中的最后一条消息
    print("================User Message================")
    user_input = request.state['messages'][-1].text
    print(user_input, "\n")

    # 2.检索知识片段, list[Document], 拼接为字符串
    docs = vector_store.similarity_search(user_input, k=2)
    context = "\n\n".join(doc.page_content for doc in docs)
    print('=============== Context ====================')
    print(context, "\n")

    # 3.拼接新系统提示词
    return f'''
    你是一个教资考试专家，你根据上下文回答用户问题，不能自己编造答案。答案要简短，直接引用原文。
    Context: {context}
    '''

# 1.初始化模型
model = init_chat_model(
    "deepseek-v4-flash",
    extra_body={'thinking': {'type': 'disabled'}}
)

# 2.定义Agent
agent = create_agent(
    model,
    middleware=[rag_prompt],
)

In [4]:
from langchain.messages import HumanMessage
# 测试
response = agent.invoke({"messages":[HumanMessage("论语中教育的目的是什么？")]})
print("==============AI Message================")
print(response["messages"][-1].text)

================User Message================
论语中教育的目的是什么？ 

=============== Context ====================
### （三）孔子及《论语》主要思想  
**教育作用**：庶、富、教；性相近，习相远。  
**教育对象**：“有教无类”，教育民主思想。  
**教育目的**：以完善人格为教育的首要目的，培养士和君子。  
**教学内容**：文、行、忠、义。  
**教学过程**：学、思、习、行。  
**教学原则**：因材施教、启发诱导、学思结合、谦虚笃实。  
**教师观**：其身正，不令而行；其身不正，虽令不从。

### （四）孟子主要思想  
**人性论**：人性本善，人先天具有仁、义、礼、智四个“善端”。  
**教育作用**：发扬善端，培养道德完人，得天下英才而教育之。  
**教学原则**：循序渐进，专心有恒。 

==============AI Message================
以完善人格为教育的首要目的，培养士和君子。


In [5]:
# 测试
response = agent.invoke({"messages":[HumanMessage("你好")]})
print("==============AI Message================")
print(response["messages"][-1].text)

================User Message================
你好 

=============== Context ====================
### （二）教育的属性  
教育的本质属性是有目的地培养人的社会活动。  
教育的永恒性。  
教育的历史性。  
教育的相对独立性。  
教育的长期性。  
教育的生产性。  
教育的民族性。

## 第二节 教育的定义  
### （一）教育的定义  
广义的教育是指一切有目的地增进人的知识和技能，发展人的智力和体力，影响人的思想品德的社会活动，具有目的性和社会性。广义教育包括社会教育、家庭教育、学校教育。广义的教育是人类社会有史以来就有的教育活动。  
狭义的教育就是指学校教育。  
教育的要素：教育者、受教育者、教育影响（主要是教育内容）。 

==============AI Message================
你好！我是教资考试专家，有什么问题可以问我。


# Agentic RAG

In [6]:
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from dotenv import load_dotenv
load_dotenv()

# 1.初始化模型
model = init_chat_model(
    "deepseek-flash",
    extra_body = {"thinking": {"type": "disabled"}}
)

# 2.定义工具
@tool
def retrieve_docs(query: str):
    """检索知识库，获取与用户提问相关的知识片段"""

    # 1.根据问题检索知识片段
    documents = vector_store.similarity_search(query)

    # 2.拼接知识片段
    context = "\n".join([doc.page_content for doc in documents])

    return context

# 3.定义Agent，绑定model和Tool
agent = create_agent(
    model,
    tools=[retrieve_docs],
    system_prompt="你是一个教资考试专家，回答与教资相关问题时，必须调用工具检索文档，根据文档回答用户的问题，不能自己编造答案。答案要简短，直接引用原文。"
)

In [7]:
# 测试
from langchain_core.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage("论语中教育的目的是什么？")]} # 查询重写
)
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

论语中教育的目的是什么？
================================== Ai Message ==================================
Tool Calls:
  retrieve_docs (call_00_szmeje31RrvlJHScjTV34968)
 Call ID: call_00_szmeje31RrvlJHScjTV34968
  Args:
    query: 论语 教育目的
================================= Tool Message =================================
Name: retrieve_docs

### （三）孔子及《论语》主要思想  
**教育作用**：庶、富、教；性相近，习相远。  
**教育对象**：“有教无类”，教育民主思想。  
**教育目的**：以完善人格为教育的首要目的，培养士和君子。  
**教学内容**：文、行、忠、义。  
**教学过程**：学、思、习、行。  
**教学原则**：因材施教、启发诱导、学思结合、谦虚笃实。  
**教师观**：其身正，不令而行；其身不正，虽令不从。
### （四）孟子主要思想  
**人性论**：人性本善，人先天具有仁、义、礼、智四个“善端”。  
**教育作用**：发扬善端，培养道德完人，得天下英才而教育之。  
**教学原则**：循序渐进，专心有恒。
### （五）荀子主要思想  
**人性论**：性恶论，化性起伪，善德是后天习得的，重视教育的作用。  
**教学原则**：学以致用，锲而不舍。
### （二）《学记》主要思想  
**教育作用**：化民成俗，其必由学；建国君民，教学为先。  
**教学相长**：教和学两方面互相影响和促进，都得到提高。  
**豫时孙摩**：包括预防性原则、及时性原则、循序渐进原则、集体教育原则。  
**长善救失**：“学者有四失，教者必知之。人之学也，或失则多，或失则寡，或失则易，或失则止。此四者，心之莫同也。知其心，然后能救其失也，教也者，长善而救其失者也。”

In [8]:
# 测试
from langchain_core.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage("你好")]}
)
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

你好
================================== Ai Message ==================================

你好！我是教资考试助手。请问有什么关于教师资格证考试的问题可以帮你？
